
This notebook follows the sequence:

1. Simulate the hotel environment  
2. Generate true revenue  
3. Inject fraud into observed revenue  
4. Estimate causal recovery models  
5. Train an expected-revenue model  
6. Standardize residuals  
7. Compute Bayesian fraud scores  
8. Evaluate fraud detection  
9. Compare with a residual-threshold benchmark  
10. Run Monte Carlo simulations  

In [1]:
# Step 0: Import packages and set the main parameters

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# -----------------------------
# Simulation settings
# -----------------------------
N_HOTELS = 50
N_DAYS = 365
TRUE_PRICE_EFFECT = 500

# Calibrated from 1,000 to 2,750.
# A larger value creates more overlap between normal and fraudulent records.
REVENUE_NOISE_SD = 2750

# Slightly higher than -3.5 to make the test-period fraud rate
# closer to the low-base-rate setting in the key-results infographic.
FRAUD_INTERCEPT = -3.35

FRAUD_MAGNITUDES = [0.05, 0.10, 0.20, 0.30]
TRAIN_RATIO = 0.70
PRIOR_FRAUD = 0.05

POSTERIOR_THRESHOLDS = [0.10, 0.20, 0.30, 0.50]
RESIDUAL_THRESHOLDS = [1.5, 2.0, 2.5, 3.0]

# Representative seeds for the step-by-step 10% example.
ENVIRONMENT_SEED = 100
REVENUE_SEED = 100
FRAUD_SEED = 190

# Teacher's required Monte Carlo repetition count.
# Change this to 10 first when testing the notebook quickly.
N_REP = 200


# Step 1: Simulate the Hotel Environment

Each row represents one hotel on one day. The simulated environment includes hotel quality, seasonality, market trend, competitor price, and hotel price.

In [2]:
def simulate_hotel_data(n_hotels=50, n_days=365, seed=42):
    """Create the fraud-free hotel business environment."""

    rng = np.random.default_rng(seed)

    # Create every hotel-day combination.
    hotel_ids = np.arange(n_hotels)
    days = np.arange(n_days)

    data = pd.MultiIndex.from_product(
        [hotel_ids, days],
        names=["hotel_id", "day"]
    ).to_frame(index=False)

    # Hotel quality is stable over time for each hotel.
    hotel_quality = rng.normal(0, 1, size=n_hotels)
    quality_map = dict(zip(hotel_ids, hotel_quality))
    data["hotel_quality"] = data["hotel_id"].map(quality_map)

    # Seasonality combines annual and weekly demand patterns.
    annual_season = 0.35 * np.sin(2 * np.pi * data["day"] / 365)
    weekly_season = 0.15 * np.sin(2 * np.pi * data["day"] / 7)
    season_noise = rng.normal(0, 0.05, size=len(data))

    data["seasonality"] = (
        annual_season
        + weekly_season
        + season_noise
    )

    # Market trend changes more slowly than weekly seasonality.
    trend_noise = rng.normal(0, 0.03, size=len(data))

    data["market_trend"] = (
        0.20 * np.sin(2 * np.pi * data["day"] / 180)
        + 0.001 * data["day"]
        + trend_noise
    )

    # Competitor price responds to market conditions and hotel quality.
    competitor_noise = rng.normal(0, 5, size=len(data))

    data["competitor_price"] = (
        100
        + 12 * data["seasonality"]
        + 8 * data["market_trend"]
        + 5 * data["hotel_quality"]
        + competitor_noise
    )

    # Hotel price is not randomly assigned.
    # It responds to the same factors that also affect revenue.
    price_noise = rng.normal(0, 5, size=len(data))

    data["price"] = (
        90
        + 10 * data["seasonality"]
        + 6 * data["market_trend"]
        + 0.45 * data["competitor_price"]
        + 4 * data["hotel_quality"]
        + price_noise
    )

    return data


hotel_data = simulate_hotel_data(
    n_hotels=N_HOTELS,
    n_days=N_DAYS,
    seed=ENVIRONMENT_SEED
)

print("Rows:", len(hotel_data))
display(hotel_data.head())


Rows: 18250


,hotel_id,day,hotel_quality,seasonality,market_trend,competitor_price,price
0,0,0,-1.15755,0.089797,-0.001467,89.160463,123.777139
1,0,1,-1.15755,0.116732,0.019455,100.528207,134.705919
2,0,2,-1.15755,0.100375,0.017326,94.013817,129.454922
3,0,3,-1.15755,0.036705,0.041525,87.915239,124.039120
4,0,4,-1.15755,0.014391,0.026020,92.328315,123.875611


# Step 2: Generate True Revenue

The true price effect is fixed at 500. Normal revenue also contains random operating variation. The larger calibrated noise makes small manipulation harder to distinguish from ordinary business variation.

In [3]:
def generate_true_revenue(
    data,
    seed=42,
    true_price_effect=500,
    revenue_noise_sd=2750
):
    """Add fraud-free revenue to the simulated hotel environment."""

    rng = np.random.default_rng(seed)
    result = data.copy()

    revenue_noise = rng.normal(
        0,
        revenue_noise_sd,
        size=len(result)
    )

    result["true_revenue"] = (
        5000
        + true_price_effect * result["price"]
        + 1200 * result["seasonality"]
        + 900 * result["market_trend"]
        + 200 * result["competitor_price"]
        + 1000 * result["hotel_quality"]
        + revenue_noise
    )

    return result


base_data = generate_true_revenue(
    hotel_data,
    seed=REVENUE_SEED,
    true_price_effect=TRUE_PRICE_EFFECT,
    revenue_noise_sd=REVENUE_NOISE_SD
)

display(base_data.head())
display(base_data["true_revenue"].describe().to_frame())


,hotel_id,day,hotel_quality,seasonality,market_trend,competitor_price,price,true_revenue
0,0,0,-1.15755,0.089797,-0.001467,89.160463,123.777139,80486.286883
1,0,1,-1.15755,0.116732,0.019455,100.528207,134.705919,92255.467283
2,0,2,-1.15755,0.100375,0.017326,94.013817,129.454922,89656.067255
3,0,3,-1.15755,0.036705,0.041525,87.915239,124.039120,85022.404125
4,0,4,-1.15755,0.014391,0.026020,92.328315,123.875611,81642.803372


,true_revenue
count,18250.000000
mean,94487.278568
std,7218.467233
min,66689.186210
25%,89527.529979
50%,94364.944831
75%,99401.538098
max,122157.522197


# Step 3: Inject Fraud into Observed Revenue

Fraud changes reported revenue, not true revenue. It is more likely during high-demand periods. Each fraud magnitude varies randomly between 60% and 140% of its target value.

In [4]:
def sigmoid(x):
    """Convert any real number into a probability between 0 and 1."""
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))


def inject_fraud(
    data,
    fraud_magnitude,
    seed=42,
    intercept=-3.35
):
    """Create observed revenue by injecting suppression or inflation fraud."""

    rng = np.random.default_rng(seed)
    result = data.copy()

    # Standardize demand variables so their effects are comparable.
    season_z = (
        result["seasonality"] - result["seasonality"].mean()
    ) / result["seasonality"].std()

    trend_z = (
        result["market_trend"] - result["market_trend"].mean()
    ) / result["market_trend"].std()

    # High-demand periods have a higher probability of fraud.
    risk_index = 1.2 * season_z + 0.8 * trend_z
    result["fraud_probability"] = sigmoid(intercept + risk_index)

    # Draw 1 for fraud and 0 for no fraud.
    result["fraud"] = rng.binomial(
        n=1,
        p=result["fraud_probability"],
        size=len(result)
    )

    # Normal records initially have observed revenue equal to true revenue.
    result["observed_revenue"] = result["true_revenue"].copy()
    result["fraud_type"] = "none"
    result["realized_magnitude"] = 0.0

    fraud_mask = result["fraud"] == 1
    number_of_fraud_records = int(fraud_mask.sum())

    if number_of_fraud_records > 0:
        # 60% suppression and 40% inflation.
        fraud_types = rng.choice(
            ["suppression", "inflation"],
            size=number_of_fraud_records,
            p=[0.60, 0.40]
        )

        # Actual magnitude varies around the target.
        realized_magnitudes = rng.uniform(
            low=fraud_magnitude * 0.60,
            high=fraud_magnitude * 1.40,
            size=number_of_fraud_records
        )

        result.loc[fraud_mask, "fraud_type"] = fraud_types
        result.loc[fraud_mask, "realized_magnitude"] = realized_magnitudes

        suppression_mask = (
            fraud_mask
            & (result["fraud_type"] == "suppression")
        )

        inflation_mask = (
            fraud_mask
            & (result["fraud_type"] == "inflation")
        )

        result.loc[suppression_mask, "observed_revenue"] = (
            result.loc[suppression_mask, "true_revenue"]
            * (
                1
                - result.loc[
                    suppression_mask,
                    "realized_magnitude"
                ]
            )
        )

        result.loc[inflation_mask, "observed_revenue"] = (
            result.loc[inflation_mask, "true_revenue"]
            * (
                1
                + result.loc[
                    inflation_mask,
                    "realized_magnitude"
                ]
            )
        )

    result["fraud_magnitude_target"] = fraud_magnitude
    result["revenue_difference"] = (
        result["observed_revenue"]
        - result["true_revenue"]
    )

    return result


In [5]:
# Create the four required fraud-magnitude scenarios.

scenario_data = {}

for i, magnitude in enumerate(FRAUD_MAGNITUDES):
    scenario_data[magnitude] = inject_fraud(
        base_data,
        fraud_magnitude=magnitude,
        seed=FRAUD_SEED,
        intercept=FRAUD_INTERCEPT
    )

fraud_summary = pd.DataFrame([
    {
        "fraud_magnitude": magnitude,
        "number_of_records": len(data),
        "number_of_fraud_records": int(data["fraud"].sum()),
        "fraud_rate": data["fraud"].mean(),
        "mean_realized_magnitude": (
            data.loc[
                data["fraud"] == 1,
                "realized_magnitude"
            ].mean()
        ),
        "mean_absolute_revenue_difference": (
            data.loc[
                data["fraud"] == 1,
                "revenue_difference"
            ].abs().mean()
        )
    }
    for magnitude, data in scenario_data.items()
])

display(fraud_summary.round(4))

# Use the 10% scenario for the detailed step-by-step example.
df_10 = scenario_data[0.10].copy()


,fraud_magnitude,number_of_records,number_of_fraud_records,fraud_rate,mean_realized_magnitude,mean_absolute_revenue_difference
0,0.05,18250,950,0.0521,0.0503,4867.4325
1,0.10,18250,950,0.0521,0.1007,9734.8649
2,0.20,18250,950,0.0521,0.2014,19469.7298
3,0.30,18250,950,0.0521,0.3021,29204.5947


# Step 4: Estimate Causal Recovery Models

The objective is to recover the known price effect of 500.

- **Naive model:** price only  
- **Adjusted model:** controls for observed market conditions  
- **Hotel fixed-effects model:** also controls for stable hotel differences through `C(hotel_id)`  

Fraud is injected into `observed_revenue`, so causal recovery is estimated using `true_revenue`.

In [6]:
def estimate_causal_recovery(data, true_price_effect=500):
    """Fit the three required causal models and compare their price estimates."""

    formulas = {
        "Naive model": (
            "true_revenue ~ price"
        ),
        "Adjusted controls": (
            "true_revenue ~ price"
            " + seasonality"
            " + market_trend"
            " + competitor_price"
        ),
        "Hotel fixed effects": (
            "true_revenue ~ price"
            " + seasonality"
            " + market_trend"
            " + competitor_price"
            " + C(hotel_id)"
        )
    }

    rows = []

    for model_name, formula in formulas.items():
        model = smf.ols(formula, data=data).fit()

        estimate = model.params["price"]
        ci_low, ci_high = model.conf_int().loc["price"]

        rows.append({
            "model": model_name,
            "estimated_price_effect": estimate,
            "true_price_effect": true_price_effect,
            "bias": estimate - true_price_effect,
            "relative_bias": (
                estimate - true_price_effect
            ) / true_price_effect,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "ci_covers_true_effect": (
                ci_low <= true_price_effect <= ci_high
            )
        })

    return pd.DataFrame(rows)


causal_results = estimate_causal_recovery(
    base_data,
    true_price_effect=TRUE_PRICE_EFFECT
)

causal_results_display = causal_results.copy()
causal_results_display["relative_bias"] *= 100

display(causal_results_display.round(3))


,model,estimated_price_effect,true_price_effect,bias,relative_bias,ci_low,ci_high,ci_covers_true_effect
0,Naive model,717.555,500,217.555,43.511,712.672,722.439,False
1,Adjusted controls,562.562,500,62.562,12.512,555.442,569.681,False
2,Hotel fixed effects,502.101,500,2.101,0.420,494.127,510.076,True


# Step 5: Train an Expected Revenue Model

The first 70% of days form the training period. The final 30% form the testing period. Auditors only observe reported revenue, so the expected-revenue model is trained using `observed_revenue`.

In [7]:
def split_train_test(data, train_ratio=0.70):
    """Split panel data by time instead of randomly mixing past and future days."""

    split_day = int(data["day"].max() * train_ratio)

    train_data = data[
        data["day"] <= split_day
    ].copy()

    test_data = data[
        data["day"] > split_day
    ].copy()

    return train_data, test_data, split_day


def add_expected_revenue(train_data, test_data):
    """Estimate normal reported revenue and calculate test-period residuals."""

    model = smf.ols(
        "observed_revenue ~ price"
        " + seasonality"
        " + market_trend"
        " + competitor_price"
        " + C(hotel_id)",
        data=train_data
    ).fit()

    result = test_data.copy()
    result["expected_revenue"] = model.predict(result)

    # Residual = what was reported minus what the model expected.
    result["residual"] = (
        result["observed_revenue"]
        - result["expected_revenue"]
    )

    return model, result


train_10, test_10, split_day = split_train_test(
    df_10,
    train_ratio=TRAIN_RATIO
)

expected_revenue_model, test_10 = add_expected_revenue(
    train_10,
    test_10
)

print("Split day:", split_day)
print("Training rows:", len(train_10))
print("Testing rows:", len(test_10))

display(
    test_10[
        [
            "hotel_id",
            "day",
            "observed_revenue",
            "expected_revenue",
            "residual",
            "fraud"
        ]
    ].head()
)


Split day: 254
Training rows: 12750
Testing rows: 5500


,hotel_id,day,observed_revenue,expected_revenue,residual,fraud
255,0,255,83427.070419,83857.912576,-430.842157,0
256,0,256,83019.426381,86723.098219,-3703.671838,0
257,0,257,79175.591371,84086.573188,-4910.981817,0
258,0,258,83703.019249,84431.714574,-728.695325,0
259,0,259,87386.667121,86696.597796,690.069325,0


# Step 6: Standardize Residuals

Median and median absolute deviation are used because they are less sensitive to fraudulent outliers than the ordinary mean and standard deviation.

In [8]:
def standardize_residuals(data):
    """Convert residuals into robust z-scores."""

    result = data.copy()
    residuals = result["residual"]

    center = np.median(residuals)
    mad = np.median(np.abs(residuals - center))
    scale = 1.4826 * mad

    # Safety fallback in the unlikely event that MAD equals zero.
    if scale == 0:
        scale = residuals.std()

    if scale == 0:
        scale = 1.0

    result["z_residual"] = (
        result["residual"] - center
    ) / scale

    result["abs_z_residual"] = (
        result["z_residual"].abs()
    )

    return result, center, scale


test_10, residual_center, residual_scale = (
    standardize_residuals(test_10)
)

print("Residual center:", round(residual_center, 3))
print("Robust residual scale:", round(residual_scale, 3))

display(
    test_10.groupby("fraud")["abs_z_residual"]
    .agg(["count", "mean", "median", "max"])
    .round(3)
)


Residual center: 9.967
Robust residual scale: 2870.073


,count,mean,median,max
fraud,,,,
0,5385,0.772,0.657,4.158
1,115,3.208,3.086,5.598


# Step 7: Compute Bayesian Fraud Scores

The Bayesian score compares how plausible a residual is under:

- normal reporting, and
- fraud involving negative or positive abnormal residuals.

The result is a probability-like audit risk score, not a final legal judgment of fraud.

In [9]:
def normal_pdf(x, mean=0, sd=1):
    """Normal probability density function."""

    return (
        np.exp(
            -0.5 * ((x - mean) / sd) ** 2
        )
        / (sd * np.sqrt(2 * np.pi))
    )


def add_bayesian_score(data, prior=0.05):
    """Convert standardized residual evidence into posterior fraud risk."""

    result = data.copy()
    z = result["z_residual"]

    likelihood_no_fraud = normal_pdf(
        z,
        mean=0,
        sd=1
    )

    likelihood_fraud_negative = normal_pdf(
        z,
        mean=-1.5,
        sd=1.8
    )

    likelihood_fraud_positive = normal_pdf(
        z,
        mean=1.5,
        sd=1.8
    )

    likelihood_fraud = (
        0.60 * likelihood_fraud_negative
        + 0.40 * likelihood_fraud_positive
    )

    numerator = likelihood_fraud * prior

    denominator = (
        numerator
        + likelihood_no_fraud * (1 - prior)
    )

    # Small constant avoids division by zero for extreme residuals.
    result["posterior_fraud_probability"] = (
        numerator / (denominator + 1e-12)
    )

    return result


test_10 = add_bayesian_score(
    test_10,
    prior=PRIOR_FRAUD
)

display(
    test_10[
        [
            "hotel_id",
            "day",
            "z_residual",
            "fraud",
            "fraud_type",
            "posterior_fraud_probability"
        ]
    ].head()
)


,hotel_id,day,z_residual,fraud,fraud_type,posterior_fraud_probability
255,0,255,-0.153588,0,none,0.020741
256,0,256,-1.293918,0,none,0.046125
257,0,257,-1.714572,0,none,0.079255
258,0,258,-0.257367,0,none,0.021331
259,0,259,0.236964,0,none,0.020310


# Step 8: Evaluate Fraud Detection

A lower threshold captures more fraud but usually creates more false positives. A higher threshold creates fewer alerts but may miss more fraud.

In [10]:
def evaluate_thresholds(
    data,
    score_column,
    thresholds,
    method_name
):
    """Calculate classification metrics at several decision thresholds."""

    y_true = data["fraud"]
    scores = data[score_column]

    auc = roc_auc_score(y_true, scores)
    average_precision = average_precision_score(
        y_true,
        scores
    )

    rows = []

    for threshold in thresholds:
        y_pred = scores >= threshold

        rows.append({
            "method": method_name,
            "threshold": threshold,
            "precision": precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "recall": recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "f1_score": f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "auc": auc,
            "average_precision": average_precision,
            "number_flagged": int(y_pred.sum()),
            "fraud_found": int(
                ((y_pred == 1) & (y_true == 1)).sum()
            ),
            "total_fraud": int(y_true.sum())
        })

    return pd.DataFrame(rows)


bayesian_eval_10 = evaluate_thresholds(
    test_10,
    score_column="posterior_fraud_probability",
    thresholds=POSTERIOR_THRESHOLDS,
    method_name="Bayesian posterior score"
)

best_bayesian_10 = bayesian_eval_10.loc[
    bayesian_eval_10["f1_score"].idxmax()
]

display(bayesian_eval_10.round(3))

print(
    "Best Bayesian threshold:",
    round(best_bayesian_10["threshold"], 3)
)
print(
    "Best Bayesian F1:",
    round(best_bayesian_10["f1_score"], 3)
)


,method,threshold,precision,recall,f1_score,auc,average_precision,number_flagged,fraud_found,total_fraud
0,Bayesian posterior score,0.1,0.306,0.887,0.455,0.977,0.754,333,102,115
1,Bayesian posterior score,0.2,0.531,0.748,0.621,0.977,0.754,162,86,115
2,Bayesian posterior score,0.3,0.700,0.670,0.684,0.977,0.754,110,77,115
3,Bayesian posterior score,0.5,0.875,0.548,0.674,0.977,0.754,72,63,115


Best Bayesian threshold: 0.3
Best Bayesian F1: 0.684


# Step 9: Add a Residual-Threshold Benchmark

This benchmark flags records using only the absolute standardized residual. It provides a simple comparison with the Bayesian risk score.

In [11]:
residual_eval_10 = evaluate_thresholds(
    test_10,
    score_column="abs_z_residual",
    thresholds=RESIDUAL_THRESHOLDS,
    method_name="Residual threshold"
)

best_residual_10 = residual_eval_10.loc[
    residual_eval_10["f1_score"].idxmax()
]

benchmark_comparison_10 = pd.DataFrame([
    best_bayesian_10,
    best_residual_10
]).reset_index(drop=True)

display(residual_eval_10.round(3))
display(benchmark_comparison_10.round(3))


,method,threshold,precision,recall,f1_score,auc,average_precision,number_flagged,fraud_found,total_fraud
0,Residual threshold,1.5,0.144,0.939,0.250,0.976,0.75,749,108,115
1,Residual threshold,2.0,0.329,0.878,0.479,0.976,0.75,307,101,115
2,Residual threshold,2.5,0.594,0.713,0.648,0.976,0.75,138,82,115
3,Residual threshold,3.0,0.886,0.539,0.670,0.976,0.75,70,62,115


,method,threshold,precision,recall,f1_score,auc,average_precision,number_flagged,fraud_found,total_fraud
0,Bayesian posterior score,0.3,0.700,0.670,0.684,0.977,0.754,110,77,115
1,Residual threshold,3.0,0.886,0.539,0.670,0.976,0.750,70,62,115


## Optional audit-prioritization check: Top 5% review pool

This is not one of the four required teacher tables, but it connects the simulation to the key-results infographic. It evaluates how much fraud is concentrated inside the highest-risk 5% of test records.

In [12]:
def summarize_top_review_pool(
    data,
    score_column,
    top_fraction=0.05
):
    """Measure audit efficiency inside the highest-risk review pool."""

    number_to_review = int(
        np.ceil(len(data) * top_fraction)
    )

    review_pool = data.nlargest(
        number_to_review,
        score_column
    )

    total_fraud = int(data["fraud"].sum())
    fraud_found = int(review_pool["fraud"].sum())

    overall_fraud_rate = data["fraud"].mean()
    review_pool_fraud_rate = review_pool["fraud"].mean()

    return pd.DataFrame([{
        "top_fraction": top_fraction,
        "records_reviewed": number_to_review,
        "fraud_captured": (
            fraud_found / total_fraud
            if total_fraud > 0
            else np.nan
        ),
        "fraud_rate_in_review_pool": (
            review_pool_fraud_rate
        ),
        "lift": (
            review_pool_fraud_rate / overall_fraud_rate
            if overall_fraud_rate > 0
            else np.nan
        ),
        "records_per_fraud_found": (
            number_to_review / fraud_found
            if fraud_found > 0
            else np.nan
        )
    }])


top5_summary_10 = summarize_top_review_pool(
    test_10,
    score_column="posterior_fraud_probability",
    top_fraction=0.05
)

display(top5_summary_10.round(3))


,top_fraction,records_reviewed,fraud_captured,fraud_rate_in_review_pool,lift,records_per_fraud_found
0,0.05,275,0.843,0.353,16.87,2.835


## Calibration check against the key-results infographic

The comparison below is diagnostic only. Exact equality is not expected because the infographic uses a richer evidence basket, whereas this notebook intentionally keeps the teacher's simpler residual-based Bayesian model.

In [13]:
calibration_comparison = pd.DataFrame({
    "metric": [
        "AUC",
        "Average precision",
        "Best F1",
        "Top-5% fraud captured",
        "Top-5% fraud rate",
        "Top-5% lift",
        "Records per fraud found"
    ],
    "infographic_target": [
        0.970,
        0.760,
        0.680,
        0.842,
        0.363,
        16.8,
        2.78
    ],
    "current_run": [
        best_bayesian_10["auc"],
        best_bayesian_10["average_precision"],
        best_bayesian_10["f1_score"],
        top5_summary_10.loc[0, "fraud_captured"],
        top5_summary_10.loc[
            0,
            "fraud_rate_in_review_pool"
        ],
        top5_summary_10.loc[0, "lift"],
        top5_summary_10.loc[
            0,
            "records_per_fraud_found"
        ]
    ]
})

calibration_comparison["difference"] = (
    calibration_comparison["current_run"]
    - calibration_comparison["infographic_target"]
)

display(calibration_comparison.round(3))


,metric,infographic_target,current_run,difference
0,AUC,0.970,0.977,0.007
1,Average precision,0.760,0.754,-0.006
2,Best F1,0.680,0.684,0.004
3,Top-5% fraud captured,0.842,0.843,0.001
4,Top-5% fraud rate,0.363,0.353,-0.010
5,Top-5% lift,16.800,16.870,0.070
6,Records per fraud found,2.780,2.835,0.055


# Step 10: Run Monte Carlo Simulations

The Monte Carlo loop uses the same functions as the detailed example, so the logic is defined only once.

Within each repetition:

- one fraud-free hotel environment is created;
- the same environment is reused across 5%, 10%, 20%, and 30% scenarios;
- only fraud magnitude changes, making the comparison cleaner;
- causal recovery is estimated once because fraud affects observed revenue, not true revenue.

In [14]:
def run_detection_pipeline(data):
    """Run Steps 5 to 7 and return the scored test data."""

    train_data, test_data, _ = split_train_test(
        data,
        train_ratio=TRAIN_RATIO
    )

    _, test_data = add_expected_revenue(
        train_data,
        test_data
    )

    test_data, _, _ = standardize_residuals(
        test_data
    )

    test_data = add_bayesian_score(
        test_data,
        prior=PRIOR_FRAUD
    )

    return test_data


def run_monte_carlo(
    n_rep=200,
    fraud_magnitudes=None
):
    """Repeat causal recovery and fraud detection across random datasets."""

    if fraud_magnitudes is None:
        fraud_magnitudes = FRAUD_MAGNITUDES

    causal_rows = []
    detection_rows = []

    for rep in range(n_rep):
        # Create one base dataset for this repetition.
        environment = simulate_hotel_data(
            n_hotels=N_HOTELS,
            n_days=N_DAYS,
            seed=1000 + rep
        )

        base = generate_true_revenue(
            environment,
            seed=5000 + rep,
            true_price_effect=TRUE_PRICE_EFFECT,
            revenue_noise_sd=REVENUE_NOISE_SD
        )

        # Causal recovery does not depend on fraud magnitude.
        causal_rep = estimate_causal_recovery(
            base,
            true_price_effect=TRUE_PRICE_EFFECT
        )

        # Copy the same causal result across magnitude labels.
        # This avoids fitting the identical causal models four times.
        for magnitude in fraud_magnitudes:
            causal_copy = causal_rep.copy()
            causal_copy["rep"] = rep
            causal_copy["fraud_magnitude"] = magnitude
            causal_rows.extend(
                causal_copy.to_dict("records")
            )

        # Detection must be rerun because observed revenue changes by magnitude.
        for magnitude in fraud_magnitudes:
            scenario = inject_fraud(
                base,
                fraud_magnitude=magnitude,
                seed=2000 + rep * 10,
                intercept=FRAUD_INTERCEPT
            )

            scored_test = run_detection_pipeline(
                scenario
            )

            bayesian_results = evaluate_thresholds(
                scored_test,
                score_column=(
                    "posterior_fraud_probability"
                ),
                thresholds=POSTERIOR_THRESHOLDS,
                method_name=(
                    "Bayesian posterior score"
                )
            )

            residual_results = evaluate_thresholds(
                scored_test,
                score_column="abs_z_residual",
                thresholds=RESIDUAL_THRESHOLDS,
                method_name="Residual threshold"
            )

            best_bayesian = bayesian_results.loc[
                bayesian_results["f1_score"].idxmax()
            ].to_dict()

            best_residual = residual_results.loc[
                residual_results["f1_score"].idxmax()
            ].to_dict()

            best_bayesian.update({
                "rep": rep,
                "fraud_magnitude": magnitude
            })

            best_residual.update({
                "rep": rep,
                "fraud_magnitude": magnitude
            })

            detection_rows.append(best_bayesian)
            detection_rows.append(best_residual)

        if (rep + 1) % 20 == 0:
            print(
                "Completed repetition:",
                rep + 1
            )

    return (
        pd.DataFrame(causal_rows),
        pd.DataFrame(detection_rows)
    )


In [15]:
# This cell may take several minutes with N_REP = 200.

causal_mc_results, detection_mc_results = (
    run_monte_carlo(
        n_rep=N_REP,
        fraud_magnitudes=FRAUD_MAGNITUDES
    )
)

print("Causal rows:", len(causal_mc_results))
print("Detection rows:", len(detection_mc_results))


Completed repetition: 20
Completed repetition: 40
Completed repetition: 60
Completed repetition: 80
Completed repetition: 100
Completed repetition: 120
Completed repetition: 140
Completed repetition: 160
Completed repetition: 180
Completed repetition: 200
Causal rows: 2400
Detection rows: 1600


# Required Output Tables

In [16]:
# Table 1: Causal recovery results

table_1_causal_recovery = (
    causal_mc_results
    .groupby(
        ["fraud_magnitude", "model"]
    )
    .agg(
        mean_estimate=(
            "estimated_price_effect",
            "mean"
        ),
        relative_bias=(
            "relative_bias",
            "mean"
        ),
        ci_coverage=(
            "ci_covers_true_effect",
            "mean"
        )
    )
    .reset_index()
)

table_1_causal_recovery[
    "relative_bias"
] *= 100

display(table_1_causal_recovery.round(3))


,fraud_magnitude,model,mean_estimate,relative_bias,ci_coverage
0,0.05,Adjusted controls,559.097,11.819,0.00
1,0.05,Hotel fixed effects,499.640,-0.072,0.93
2,0.05,Naive model,714.401,42.880,0.00
3,0.10,Adjusted controls,559.097,11.819,0.00
4,0.10,Hotel fixed effects,499.640,-0.072,0.93
5,0.10,Naive model,714.401,42.880,0.00
6,0.20,Adjusted controls,559.097,11.819,0.00
7,0.20,Hotel fixed effects,499.640,-0.072,0.93
8,0.20,Naive model,714.401,42.880,0.00
9,0.30,Adjusted controls,559.097,11.819,0.00


In [17]:
# Table 2: Bayesian fraud detection under 10% manipulation

table_2_bayesian_10pct = (
    bayesian_eval_10[
        [
            "threshold",
            "precision",
            "recall",
            "f1_score",
            "auc",
            "average_precision",
            "number_flagged",
            "fraud_found"
        ]
    ]
    .copy()
)

display(table_2_bayesian_10pct.round(3))


,threshold,precision,recall,f1_score,auc,average_precision,number_flagged,fraud_found
0,0.1,0.306,0.887,0.455,0.977,0.754,333,102
1,0.2,0.531,0.748,0.621,0.977,0.754,162,86
2,0.3,0.700,0.670,0.684,0.977,0.754,110,77
3,0.5,0.875,0.548,0.674,0.977,0.754,72,63


In [18]:
# Table 3: Detection performance across fraud magnitudes

table_3_detection_across_magnitudes = (
    detection_mc_results[
        detection_mc_results["method"]
        == "Bayesian posterior score"
    ]
    .groupby("fraud_magnitude")
    .agg(
        mean_best_threshold=("threshold", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_best_f1=("f1_score", "mean"),
        mean_auc=("auc", "mean"),
        mean_average_precision=(
            "average_precision",
            "mean"
        )
    )
    .reset_index()
)

display(
    table_3_detection_across_magnitudes
    .round(3)
)


,fraud_magnitude,mean_best_threshold,mean_precision,mean_recall,mean_best_f1,mean_auc,mean_average_precision
0,0.05,0.213,0.292,0.256,0.261,0.788,0.201
1,0.10,0.417,0.799,0.641,0.706,0.965,0.757
2,0.20,0.500,0.911,0.981,0.944,1.000,0.993
3,0.30,0.500,0.907,0.936,0.921,0.974,0.956


In [19]:
# Table 4: Benchmark comparison under 10% manipulation

table_4_benchmark_10pct = (
    benchmark_comparison_10[
        [
            "method",
            "threshold",
            "precision",
            "recall",
            "f1_score",
            "auc",
            "average_precision",
            "number_flagged",
            "fraud_found"
        ]
    ]
    .copy()
)

display(table_4_benchmark_10pct.round(3))


,method,threshold,precision,recall,f1_score,auc,average_precision,number_flagged,fraud_found
0,Bayesian posterior score,0.3,0.700,0.670,0.684,0.977,0.754,110,77
1,Residual threshold,3.0,0.886,0.539,0.670,0.976,0.750,70,62


# Save the Main Results

The code below saves the four required tables, the top-5% result, and the calibration comparison as CSV files.

In [20]:
from pathlib import Path
import zipfile

output_folder = Path(
    "hotel_fraud_simulation_outputs"
)

output_folder.mkdir(
    exist_ok=True
)

table_1_causal_recovery.to_csv(
    output_folder / "table_1_causal_recovery.csv",
    index=False
)

table_2_bayesian_10pct.to_csv(
    output_folder / "table_2_bayesian_10pct.csv",
    index=False
)

table_3_detection_across_magnitudes.to_csv(
    output_folder
    / "table_3_detection_across_magnitudes.csv",
    index=False
)

table_4_benchmark_10pct.to_csv(
    output_folder / "table_4_benchmark_10pct.csv",
    index=False
)

top5_summary_10.to_csv(
    output_folder / "top5_review_pool_10pct.csv",
    index=False
)

calibration_comparison.to_csv(
    output_folder / "calibration_comparison.csv",
    index=False
)

zip_path = Path(
    "hotel_fraud_simulation_outputs.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:
    for file_path in output_folder.glob("*.csv"):
        zip_file.write(
            file_path,
            arcname=file_path.name
        )

print("Saved:", zip_path.resolve())


Saved: /content/hotel_fraud_simulation_outputs.zip
